# One JRC flood raster: exact Folium viewer

Set one TIFF path below and run the two code cells. The notebook reads and draws only that file. Each flooded source pixel is transformed to a Folium polygon; there is no raster-image overlay, preview-grid, catalogue scan, or widget dependency.

For browser safety, events with more than `MAX_NATIVE_PIXELS` flooded cells stop with an error instead of being approximated.

In [ ]:
from pathlib import Path

import folium
import matplotlib as mpl
import numpy as np
import rasterio
from pyproj import Transformer
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change only this line to open another JRC flood-depth TIFF.
RASTER_PATH = PROJECT_ROOT / 'data' / 'JRC_flood_depth_maps' / '2015' / 'WD_MERGE_2014-12-29---2015-01-05_duration_7_days_cluster_004_A0_000005_A_000006_lat_05570_lon_02013_size_0120.tif'
MAX_NATIVE_PIXELS = 20_000

if not RASTER_PATH.is_file():
    raise FileNotFoundError(f'Raster not found: {RASTER_PATH}')

print(RASTER_PATH)

In [ ]:
pixels = []
with rasterio.open(RASTER_PATH) as src:
    source_transform = src.transform
    source_crs = src.crs
    nodata = src.nodata

    for _, window in src.block_windows(1):
        block = src.read(1, window=window, masked=True).astype(np.float32, copy=False)
        values = block.data
        invalid = np.ma.getmaskarray(block) | (values <= 0) | (values == 9999)
        if nodata is not None:
            invalid |= values == nodata
        rows, cols = np.where(~invalid)
        if len(pixels) + rows.size > MAX_NATIVE_PIXELS:
            raise ValueError(
                f'More than {MAX_NATIVE_PIXELS:,} flooded native pixels. '
                'This strict notebook refuses to approximate the event.'
            )
        pixels.extend(
            (int(window.row_off + row), int(window.col_off + col), float(values[row, col]))
            for row, col in zip(rows, cols)
        )

if not pixels:
    raise ValueError('No positive flood-depth pixels found.')

depths = np.array([value for _, _, value in pixels])
vmin = float(depths.min())
vmax = float(np.quantile(depths, 0.995))
if vmax <= vmin:
    vmax = float(depths.max())
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
cmap = mpl.colormaps['turbo']
to_latlon = Transformer.from_crs(source_crs, 'EPSG:4326', always_xy=True)

flood_map = folium.Map(tiles='CartoDB positron', prefer_canvas=True)
latitudes, longitudes = [], []
for row, col, depth in pixels:
    left, top = source_transform * (col, row)
    right, bottom = source_transform * (col + 1, row + 1)
    lons, lats = to_latlon.transform([left, right, right, left], [top, top, bottom, bottom])
    latitudes.extend(lats)
    longitudes.extend(lons)
    folium.Polygon(
        locations=list(zip(lats, lons)),
        stroke=False,
        fill=True,
        fill_color=mpl.colors.to_hex(cmap(norm(depth))),
        fill_opacity=0.9,
        tooltip=f'Depth: {depth:.1f} cm',
    ).add_to(flood_map)

flood_map.fit_bounds([[min(latitudes), min(longitudes)], [max(latitudes), max(longitudes)]])
print(f'Drawn {len(pixels):,} exact native flood pixels; depth range: {vmin:.1f}–{vmax:.1f} cm')
display(flood_map)